# Agenda 

The goal of this notebook is to ensure eveything is setup for the workshop

## Setup

In [ ]:
%pip install -r ../requirements.txt

## Imports

In [ ]:
import dotenv
import os
import requests
import rich

In [ ]:
dotenv.load_dotenv("../.env", override=True)

In [ ]:
env_vars_to_check = [
    "OPENAI_API_KEY",
    "OPENAI_BASE_URL",
    "OPENAI_MODEL",
    "TAVILY_API_KEY",
    "TAVILY_BASE_URL",
    "PHOENIX_COLLECTOR_ENDPOINT",
    "PHOENIX_PROJECT_NAME",
]

def _mask(value: str, keep: int = 4) -> str:
    if value is None:
        return "<NOT SET>"
    if len(value) <= keep * 2:
        return "*" * len(value)
    return f"{value[:keep]}...{value[-keep:]}"


for name in env_vars_to_check:
    value = os.environ.get(name)
    globals()[name] = value
    if "KEY" in name and value is not None:
        print(f"{name}={_mask(value)}")
    else:
        print(f"{name}={value if value is not None else '<NOT SET>'}")


## OpenAI validation

In [ ]:
from langchain_openai import ChatOpenAI

In [ ]:
assert OPENAI_BASE_URL, "OPENAI_BASE_URL is not set"
assert OPENAI_API_KEY, "OPENAI_API_KEY is not set"

models_url = f"{OPENAI_BASE_URL.rstrip('/')}/models"

headers = {"Authorization": f"Bearer {OPENAI_API_KEY}"} 
resp = requests.get(models_url, headers=headers, timeout=30)
resp.raise_for_status()

models = [m.get("id", "<unknown>") for m in resp.json().get("data", [])]

print(f"Found {len(models)} model(s):")
for m in models:
    print("-", m)

In [ ]:

llm = ChatOpenAI(
    model=os.environ.get("OPENAI_MODEL", models[0] if models else "gpt-4o-mini"),
    base_url=OPENAI_BASE_URL,
    api_key=OPENAI_API_KEY,
    #temperature=0.2,
    #max_tokens=512,
)

llm.invoke("what is the weather in seattle")

## Tavilly



[Tavily](https://www.tavily.com/) is a service provider that enables agents to access the web

In [ ]:
def tavily_search(query, **kw):
    if TAVILY_API_KEY is None:
        r = requests.post(TAVILY_BASE_URL,
                        json={"query": query, **kw}, timeout=60)
        r.raise_for_status()
        return r.json()
    else:
        from tavily import TavilyClient
        tavily_client = TavilyClient(api_key=TAVILY_API_KEY)
        return tavily_client.search(query=query, **kw)


res = tavily_search("what is the weather in seattle")
print(res)


In [ ]:
rich.print(res)

In [ ]:
res = tavily_search("what is the best running shoes")
rich.print(res)

## LLM Monitoring with Arize Phoenix OTEL

It can be hard to monitor LLM calls especially when they are part of a larger workflow.

We will set up [Arize Phoenix](https://phoenix.arize.com/) OpenTelemetry to help with that.



In [ ]:
import subprocess
import time

if PHOENIX_COLLECTOR_ENDPOINT:
    print(f"Phoenix endpoint already configured: {PHOENIX_COLLECTOR_ENDPOINT}")
else:
    print("Starting local Phoenix server...")
    subprocess.Popen(
        ["python", "-m", "phoenix.server.main", "serve"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
    time.sleep(4)  # wait for server to be ready
    PHOENIX_COLLECTOR_ENDPOINT = "http://localhost:6006"
    os.environ["PHOENIX_COLLECTOR_ENDPOINT"] = PHOENIX_COLLECTOR_ENDPOINT
    print("Phoenix UI: http://localhost:6006")
    print("PHOENIX_COLLECTOR_ENDPOINT set to: http://localhost:6006")


In [ ]:
PHOENIX_PROJECT_NAME

In [ ]:
from phoenix.otel import register
from opentelemetry import trace

from openinference.instrumentation.langchain import LangChainInstrumentor
if PHOENIX_COLLECTOR_ENDPOINT:
    # configure the Phoenix tracer
    tracer_provider = register(
        project_name=PHOENIX_PROJECT_NAME, 
        auto_instrument=False 
    )
else:
    tracer_provider = trace.NoOpTracerProvider()

LangChainInstrumentor().instrument(tracer_provider=tracer_provider)
tracer = trace.get_tracer(__name__)


In [ ]:
from langchain.agents import create_agent

In [ ]:

from langchain_openai import ChatOpenAI

def get_weather(city: str) -> str:  
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"

llm = ChatOpenAI(
    model=OPENAI_MODEL if OPENAI_MODEL else models[0],
    base_url=OPENAI_BASE_URL,
    api_key=OPENAI_API_KEY
)

agent = create_agent(
    llm,
    tools=[get_weather],  
    system_prompt="You are a helpful assistant"  
)

res = agent.invoke(
    {"messages": [{"role": "user", "content": "what is the weather in seattle"}]}
)


In [ ]:
if not PHOENIX_COLLECTOR_ENDPOINT:
    print("Phoenix tracing is disabled (no PHOENIX_COLLECTOR_ENDPOINT set). Skipping trace link.")
else:
    arize_project_response = requests.get(
        f"{PHOENIX_COLLECTOR_ENDPOINT}/v1/projects",
    )
    data = arize_project_response.json()
    project = next((p for p in data.get("data", []) if p.get("name") == PHOENIX_PROJECT_NAME), None)
    if project:
        print(f"View your traces at:\n{PHOENIX_COLLECTOR_ENDPOINT}/projects/{project['id']}/spans")
    else:
        print(f"Project '{PHOENIX_PROJECT_NAME}' not found yet — run the agent cell first, then re-run this cell.")
        print(f"Phoenix UI: {PHOENIX_COLLECTOR_ENDPOINT}")


Trace of above call
![Trace of above call](../images/trace_setup_weather.png)
